# Grant the app access to your team schema

Run this right after `databricks bundle deploy`, before you start the app.

It looks up the service principal of `<team>-sales-agent` and grants it `USE SCHEMA`, `SELECT` and `EXECUTE` on `<catalog>.<team>`. The facilitator has already granted `USE CATALOG` on the catalog to `account users`.

Who can run it:
* Looking up the app's service principal needs at least `CAN_VIEW` on the app. The teammate who deployed the bundle has `CAN_MANAGE`.
* Granting on the schema needs schema ownership or `MANAGE` on the schema. The teammate who ran the seed notebook owns the schema.

If you have neither, the notebook stops and tells you what's missing. Ask that teammate or a facilitator instead of trying another route.

In [0]:
dbutils.widgets.text("catalog", "adb_fgfworkshop")
dbutils.widgets.text("team", "team8")

catalog = dbutils.widgets.get("catalog").strip()
team = dbutils.widgets.get("team").strip()
if not catalog or not team:
    raise ValueError("Set the catalog and team widgets first, for example team = team1.")

schema = f"{catalog}.{team}"
app_name = f"{team}-sales-agent"
print(f"App: {app_name}\nSchema: {schema}")

In [0]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.errors import NotFound, PermissionDenied

w = WorkspaceClient()
me = w.current_user.me().user_name

try:
    app = w.apps.get(app_name)
except NotFound:
    raise RuntimeError(f"App {app_name} does not exist yet. Run `databricks bundle deploy --var team={team} ...` first.") from None
except PermissionDenied:
    raise RuntimeError(
        f"{me} cannot view app {app_name}. Ask the teammate who deployed the bundle to run this notebook, "
        "or to give you CAN_MANAGE on the app."
    ) from None

sp_id = app.service_principal_client_id
if not sp_id:
    raise RuntimeError(f"App {app_name} has no service principal yet. Wait for the deploy to finish, then re-run.")
print(f"Service principal: {app.service_principal_name} ({sp_id})")

In [0]:
privileges = ["USE SCHEMA", "SELECT", "EXECUTE"]
failed = []

for privilege in privileges:
    statement = f"GRANT {privilege} ON SCHEMA {schema} TO `{sp_id}`"
    try:
        spark.sql(statement)
        print(f"OK      {statement}")
    except Exception as exc:
        failed.append(privilege)
        print(f"FAILED  {statement}\n        {str(exc).splitlines()[0][:300]}")
statement = f"GRANT USE CATALOG ON CATALOG {catalog} TO `{sp_id}`" 
spark.sql(statement)
print(f"OK      {statement}")
if failed:
    raise RuntimeError(
        f"{me} could not grant {', '.join(failed)} on {schema}. This needs schema ownership or MANAGE on the schema. "
        "Ask the teammate who ran the seed notebook (the schema owner) or a facilitator to run this notebook."
    )

In [0]:
granted = {row["ActionType"] for row in spark.sql(f"SHOW GRANTS `{sp_id}` ON SCHEMA {schema}").collect()}
missing = [p for p in privileges if p not in granted]
print(f"Granted to {sp_id} on {schema}: {sorted(granted)}")

try:
    catalog_grants = spark.sql(f"SHOW GRANTS ON CATALOG {catalog}").collect()
    if not any(r["Principal"] == "account users" and r["ActionType"] in ("USE CATALOG", "ALL PRIVILEGES") for r in catalog_grants):
        print(f"Warning: no USE CATALOG for `account users` on {catalog}. Ask a facilitator to grant it, or the app cannot see the schema.")
except Exception:
    print(f"Could not read grants on catalog {catalog}; assuming the facilitator granted USE CATALOG to `account users`.")

if missing:
    raise RuntimeError(f"Still missing: {missing}")
print("Done. Next: databricks bundle run sales_agent --var team=... --var warehouse_id=...")